In [0]:
# ============================================================
# PROYECTO FINAL ELECTROCASA
# 08. SILVER - SUCURSALES
# 01. LECTURA Y EXPLORACIÓN DE BRONZE
# ============================================================

CATALOGO = "proyecto_final"

TABLA_ORIGEN = f"{CATALOGO}.bronze.sucursales"
TABLA_DESTINO = f"{CATALOGO}.silver.sucursales"
TABLA_CUARENTENA = f"{CATALOGO}.silver.cuarentena"

# Leer los datos originales de Bronze
df_bronze = spark.table(TABLA_ORIGEN)

print("ORIGEN:", TABLA_ORIGEN)
print("DESTINO SILVER:", TABLA_DESTINO)
print("CUARENTENA:", TABLA_CUARENTENA)

print("\nESQUEMA DE BRONZE:")
df_bronze.printSchema()

print("\nTOTAL DE REGISTROS EN BRONZE:")
print(df_bronze.count())

print("\nMUESTRA DE REGISTROS:")
display(df_bronze.limit(10))

In [0]:
from pyspark.sql.functions import to_json, struct, col

# Guardar el registro original de Bronze como JSON
df_bronze_original = df_bronze.withColumn(
    "_datos_originales",
    to_json(
        struct(*[col(c) for c in df_bronze.columns])
    )
)

In [0]:
# ============================================================
# 02. NORMALIZACIÓN DE DATOS - SUCURSALES
# Proyecto: ElectroCasa
# Capa: Silver
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    initcap,
    when
)

df_normalizado = (
    df_bronze_original

    # Eliminar espacios al inicio y final
    .withColumn("sucursal_id", trim(col("sucursal_id")))
    .withColumn("nombre_sucursal", trim(col("nombre_sucursal")))
    .withColumn("distrito", trim(col("distrito")))
    .withColumn("ciudad", trim(col("ciudad")))
    .withColumn("region", trim(col("region")))

    # Normalizar nombres de ciudades
    .withColumn("ciudad", initcap(col("ciudad")))

    # Corregir ciudad sin tilde
    .withColumn(
        "ciudad",
        when(
            col("ciudad") == "Huanuco",
            "Huánuco"
        ).otherwise(col("ciudad"))
    )
)

print("TOTAL DE REGISTROS NORMALIZADOS:")
print(df_normalizado.count())

print("\nCIUDADES DESPUÉS DE NORMALIZAR:")

display(
    df_normalizado
    .groupBy("ciudad")
    .count()
    .orderBy("ciudad")
)

In [0]:
# ============================================================
# 03. VALIDACIÓN DE CALIDAD - SUCURSALES
# Proyecto: ElectroCasa
# Capa: Silver
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    lit,
    current_date,
    when,
    concat_ws,
    array
)

# Validación de campos obligatorios
campos_validos = (
    col("sucursal_id").isNotNull()
    & (trim(col("sucursal_id")) != "")
    & col("nombre_sucursal").isNotNull()
    & (trim(col("nombre_sucursal")) != "")
    & col("distrito").isNotNull()
    & (trim(col("distrito")) != "")
    & col("ciudad").isNotNull()
    & (trim(col("ciudad")) != "")
    & col("region").isNotNull()
    & (trim(col("region")) != "")
    & col("fecha_apertura").isNotNull()
)

# Validación de fecha
fecha_valida = (
    col("fecha_apertura").isNotNull()
    & (col("fecha_apertura") >= lit("1900-01-01").cast("date"))
    & (col("fecha_apertura") <= current_date())
)

# Clasificación y motivos de rechazo
df_validado = (
    df_normalizado
    .withColumn(
        "motivo_rechazo",
        concat_ws(
            ", ",
            array(
                when(
                    ~campos_validos,
                    lit("CAMPO_OBLIGATORIO_INVALIDO")
                ),
                when(
                    ~fecha_valida,
                    lit("FECHA_APERTURA_INVALIDA")
                )
            )
        )
    )
)

# Separar válidos e inválidos
df_validos = df_validado.filter(
    col("motivo_rechazo") == ""
).drop("motivo_rechazo")

df_rechazados = df_validado.filter(
    col("motivo_rechazo") != ""
)

# Mostrar resultados
print("===== RESULTADOS DE CALIDAD =====")

print("Total normalizados:", df_normalizado.count())
print("Registros válidos:", df_validos.count())
print("Registros rechazados:", df_rechazados.count())

print("\nREGISTROS RECHAZADOS:")
display(df_rechazados)

In [0]:
# ============================================================
# 04. CONTROL DE DUPLICADOS - SUCURSALES
# Proyecto: ElectroCasa
# Capa: Silver
# ============================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    row_number,
    lit,
    when
)

# Criterio determinista:
# conservar el registro con la fecha de ingesta más reciente.
# Los demás campos resuelven posibles empates.
ventana = (
    Window
    .partitionBy("sucursal_id")
    .orderBy(
        col("fecha_ingesta").desc_nulls_last(),
        col("fecha_apertura").desc_nulls_last(),
        col("nombre_sucursal").asc_nulls_last(),
        col("distrito").asc_nulls_last(),
        col("ciudad").asc_nulls_last(),
        col("region").asc_nulls_last(),
        col("archivo_origen").asc_nulls_last()
    )
)

df_numerado = df_validos.withColumn(
    "numero_fila",
    row_number().over(ventana)
)

# Registros únicos para Silver
df_silver = (
    df_numerado
    .filter(col("numero_fila") == 1)
    .drop("numero_fila")
)

# Duplicados excedentes para cuarentena
df_duplicados = (
    df_numerado
    .filter(col("numero_fila") > 1)
    .drop("numero_fila")
    .withColumn(
        "motivo_rechazo",
        lit("SUCURSAL_DUPLICADA")
    )
)

# Unir todos los rechazados
df_cuarentena_sucursales = df_rechazados.unionByName(
    df_duplicados
)

print("===== CONTROL DE DUPLICADOS =====")

print("Registros válidos antes de deduplicar:", df_validos.count())
print("Registros únicos para Silver:", df_silver.count())
print("Registros duplicados:", df_duplicados.count())
print("Total para cuarentena:", df_cuarentena_sucursales.count())

display(df_duplicados)

In [0]:
# ============================================================
# 05. CREACIÓN DE TABLA DE CUARENTENA
# Proyecto: ElectroCasa
# Capa: Silver
# ============================================================

spark.sql("""
    CREATE TABLE IF NOT EXISTS proyecto_final.silver.cuarentena (
        fuente STRING,
        clave_registro STRING,
        motivo_rechazo STRING,
        fecha_rechazo TIMESTAMP,
        datos_originales STRING,
        archivo_origen STRING
    )
    USING DELTA
""")

print("Tabla de cuarentena creada o existente.")

print("\nESQUEMA DE CUARENTENA:")

spark.sql("""
    DESCRIBE TABLE proyecto_final.silver.cuarentena
""").show(truncate=False)

In [0]:
%sql
DESCRIBE TABLE proyecto_final.silver.cuarentena;

In [0]:
# ============================================================
# 08. ESCRITURA DE DATOS - SILVER SUCURSALES
# Proyecto: ElectroCasa
# ============================================================

from pyspark.sql.functions import col

TABLA_DESTINO = "proyecto_final.silver.sucursales"

# Seleccionar las columnas definitivas del contrato
# No incluir el JSON auxiliar ni los metadatos de Bronze
df_silver_final = df_silver.select(
    col("sucursal_id"),
    col("nombre_sucursal"),
    col("distrito"),
    col("ciudad"),
    col("region"),
    col("fecha_apertura")
)

# Validación previa a la escritura
total_bronze = df_bronze.count()
total_validos = df_silver_final.count()
total_rechazados = df_cuarentena_sucursales.count()

print("===== CONTROL PREVIO =====")
print("Registros Bronze:", total_bronze)
print("Registros válidos:", total_validos)
print("Registros rechazados:", total_rechazados)

# Verificar que no se pierdan registros
assert total_bronze == total_validos + total_rechazados, \
    "ERROR: No coincide el total de registros."

assert total_validos > 0, \
    "ERROR: No hay registros válidos para escribir."

# Escritura en Delta
(
    df_silver_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLA_DESTINO)
)

print("\nTabla Silver creada correctamente.")
print("DESTINO:", TABLA_DESTINO)

In [0]:
%sql
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sucursal_id) AS sucursales_distintas,
    COUNT(DISTINCT ciudad) AS ciudades_distintas,
    MIN(fecha_apertura) AS fecha_minima,
    MAX(fecha_apertura) AS fecha_maxima
FROM proyecto_final.silver.sucursales;

In [0]:
%sql
SELECT
    sucursal_id,
    nombre_sucursal,
    ciudad,
    region
FROM proyecto_final.silver.sucursales
WHERE ciudad IN ('Arequipa', 'Piura', 'Huánuco')
ORDER BY ciudad, sucursal_id;

In [0]:
# ============================================================
# 11. ESCRITURA EN CUARENTENA - SUCURSALES
# Proyecto: ElectroCasa
# Capa: Silver
# ============================================================

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp
)

TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

# Adaptar los rechazos a la estructura centralizada
df_cuarentena_final = (
    df_cuarentena_sucursales
    .select(
        lit("sucursales").alias("fuente"),
        col("sucursal_id").alias("clave_registro"),
        col("motivo_rechazo"),
        current_timestamp().alias("fecha_rechazo"),
        col("_datos_originales").alias("datos_originales"),
        col("archivo_origen")
    )
)

total_rechazados = df_cuarentena_final.count()

print("Registros para cuarentena:", total_rechazados)

if total_rechazados > 0:

    (
        df_cuarentena_final.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_CUARENTENA)
    )

    print("Registros rechazados almacenados en cuarentena.")

else:

    print("No existen registros rechazados de sucursales.")
    print("No se requiere escritura en cuarentena.")

In [0]:
%sql
SELECT
    fuente,
    COUNT(*) AS total_rechazados
FROM proyecto_final.silver.cuarentena
GROUP BY fuente
ORDER BY fuente;

In [0]:
%sql
SELECT
    sucursal_id,
    nombre_sucursal,
    ciudad
FROM proyecto_final.silver.sucursales
WHERE LOWER(nombre_sucursal) NOT LIKE '%electrocasa%'
ORDER BY sucursal_id;